# 02 — AGT-A Authorization

Deep dive into role-based access control using `authorization.yaml`.

In [ ]:
import sys; sys.path.insert(0, '..')
from src.agents.customer_service import CustomerServiceAgent
from src.models import AgentRole

## PII Access: Consent Check

In [ ]:
# ALLOWED: customer_service role + consent=True (CUST-001)
cs = CustomerServiceAgent(role=AgentRole.CUSTOMER_SERVICE)
result = cs.invoke_tool('access_customer_pii', {'customer_id': 'CUST-001'})
print('ALLOWED?', result.get('ok', True) is not False)
print(result)

In [ ]:
# DENIED: customer_service role + consent=False (CUST-002)
result = cs.invoke_tool('access_customer_pii', {'customer_id': 'CUST-002'})
print('DENIED?', result.get('ok') is False)
print(result)

In [ ]:
# DENIED: cashier role (regardless of consent)
cashier = CustomerServiceAgent(role=AgentRole.CASHIER)
result = cashier.invoke_tool('access_customer_pii', {'customer_id': 'CUST-001'})
print('DENIED?', result.get('ok') is False)
print(result)

## Tier Updates

In [ ]:
# DENIED: cashier cannot update
result = cashier.invoke_tool('update_customer_tier', {'customer_id': 'CUST-001', 'new_tier': 'platinum', 'reason': 'test'})
print(result)

In [ ]:
# ALLOWED: customer_service can update
result = cs.invoke_tool('update_customer_tier', {'customer_id': 'CUST-001', 'new_tier': 'silver', 'reason': 'test'})
print(result)

## Order Amount Caps

In [ ]:
from src.agents.order_processing import OrderProcessingAgent
op_cashier = OrderProcessingAgent(role=AgentRole.CASHIER)
op_mgr = OrderProcessingAgent(role=AgentRole.STORE_MANAGER)

print('Cashier $100:', op_cashier.invoke_tool('process_order', {'customer_id': 'CUST-001', 'amount': 100.0}))
print('Cashier $600:', op_cashier.invoke_tool('process_order', {'customer_id': 'CUST-001', 'amount': 600.0}))
print('Manager $600:', op_mgr.invoke_tool('process_order', {'customer_id': 'CUST-001', 'amount': 600.0}))